# Multilingual Health QA — Fine-tuning Pipeline
**Competition:** Zindi Multilingual Health Question Answering in Low-Resource African Languages  
**Model:** google/mt5-small + LoRA (PEFT)  
**Languages:** Akan, Amharic, Luganda, Swahili, English

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Innocent-Gershon/multilingual-health-qa/blob/main/notebooks/02_Training_and_Inference.ipynb)

## Step 1 — Install Dependencies

In [ ]:
!pip install -q transformers==4.44.0 peft==0.12.0 rouge-score sentencepiece protobuf accelerate datasets

## Step 2 — Upload Data Files

In [ ]:
from google.colab import files
uploaded = files.upload()  # upload Train.csv, Val.csv, Test.csv

## Step 3 — Configuration

In [ ]:
import os, random, numpy as np, torch

# ── Experiment 2: mt5-base, fixed generation ──
CFG = {
    'model_name'      : 'google/mt5-base',
    'max_input_length': 128,
    'max_target_length': 256,
    'batch_size'      : 4,
    'grad_accum'      : 8,
    'lr'              : 3e-4,
    'epochs'          : 3,
    'warmup_steps'    : 200,
    'lora_r'          : 16,
    'lora_alpha'      : 32,
    'lora_dropout'    : 0.1,
    'num_beams'       : 4,
    'seed'            : 42,
    'output_dir'      : 'outputs/checkpoint',
    'submission_path' : 'submission.csv',
}

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(CFG['seed'])
os.makedirs(CFG['output_dir'], exist_ok=True)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)
if device == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))

## Step 4 — Load and Preprocess Data

In [ ]:
import pandas as pd, re

SUBSET_TO_LANG = {
    'Aka_Gha': 'Akan', 'Amh_Eth': 'Amharic',
    'Lug_Uga': 'Luganda', 'Swa_Ken': 'Swahili',
    'Eng_Uga': 'English', 'Eng_Gha': 'English',
    'Eng_Eth': 'English', 'Eng_Ken': 'English',
}

def clean(text):
    return re.sub(r'\s+', ' ', str(text).strip())

def make_prompt(row):
    lang = SUBSET_TO_LANG.get(row['subset'], 'English')
    return f"answer health question in {lang}: {clean(row['input'])}"

def load_data(path, is_test=False):
    df = pd.read_csv(path)
    df = df[df['input'].notna() & (df['input'].str.strip() != '')]
    if not is_test:
        df = df[df['output'].notna() & (df['output'].str.strip() != '')]
        df['output'] = df['output'].apply(clean)
    df['prompt'] = df.apply(make_prompt, axis=1)
    return df.reset_index(drop=True)

train_df = load_data('Train.csv')
val_df   = load_data('Val.csv')
test_df  = load_data('Test.csv', is_test=True)

print(f'Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}')
print('Sample prompt :', train_df['prompt'].iloc[0])
print('Sample output :', train_df['output'].iloc[0][:150])

## Step 5 — Tokenizer and Model with LoRA

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from peft import get_peft_model, LoraConfig, TaskType

tokenizer = AutoTokenizer.from_pretrained(CFG['model_name'])
base_model = AutoModelForSeq2SeqLM.from_pretrained(CFG['model_name'])

lora_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=CFG['lora_r'],
    lora_alpha=CFG['lora_alpha'],
    lora_dropout=CFG['lora_dropout'],
    target_modules=['q', 'v'],
)
model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

## Step 6 — Tokenize Dataset (Fixed)

In [ ]:
from torch.utils.data import Dataset

class QADataset(Dataset):
    def __init__(self, df, tokenizer, max_input, max_target, is_test=False):
        self.prompts  = df['prompt'].tolist()
        self.outputs  = df['output'].tolist() if not is_test else None
        self.tokenizer = tokenizer
        self.max_input  = max_input
        self.max_target = max_target
        self.is_test    = is_test

    def __len__(self):
        return len(self.prompts)

    def __getitem__(self, idx):
        # tokenize input
        model_inputs = self.tokenizer(
            self.prompts[idx],
            max_length=self.max_input,
            truncation=True,
        )
        if not self.is_test:
            # tokenize target — use text_target to avoid prefix issues
            with self.tokenizer.as_target_tokenizer():
                labels = self.tokenizer(
                    self.outputs[idx],
                    max_length=self.max_target,
                    truncation=True,
                )
            # replace pad token id with -100 so loss ignores padding
            label_ids = [
                -100 if t == self.tokenizer.pad_token_id else t
                for t in labels['input_ids']
            ]
            model_inputs['labels'] = label_ids
        return model_inputs

train_dataset = QADataset(train_df, tokenizer, CFG['max_input_length'], CFG['max_target_length'])
val_dataset   = QADataset(val_df,   tokenizer, CFG['max_input_length'], CFG['max_target_length'])

# Quick sanity check — labels must NOT be all -100
sample = train_dataset[0]
valid_labels = [l for l in sample['labels'] if l != -100]
print(f'Sample label tokens (non-padding): {len(valid_labels)} tokens')
print('Decoded sample label:', tokenizer.decode(valid_labels))

## Step 7 — Training

In [ ]:
from transformers import (
    Seq2SeqTrainer, Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq
)
from rouge_score import rouge_scorer as rs

collator = DataCollatorForSeq2Seq(
    tokenizer, model=model, padding=True, pad_to_multiple_of=8
)

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]
    decoded_preds  = tokenizer.batch_decode(preds, skip_special_tokens=True)
    labels         = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    scorer = rs.RougeScorer(['rouge1', 'rougeL'], use_stemmer=False)
    r1, rl = [], []
    for p, l in zip(decoded_preds, decoded_labels):
        s = scorer.score(l, p)
        r1.append(s['rouge1'].fmeasure)
        rl.append(s['rougeL'].fmeasure)
    return {
        'rouge1_f1': round(sum(r1)/len(r1), 4),
        'rougeL_f1': round(sum(rl)/len(rl), 4),
    }

training_args = Seq2SeqTrainingArguments(
    output_dir=CFG['output_dir'],
    num_train_epochs=CFG['epochs'],
    per_device_train_batch_size=CFG['batch_size'],
    per_device_eval_batch_size=CFG['batch_size'],
    gradient_accumulation_steps=CFG['grad_accum'],
    learning_rate=CFG['lr'],
    warmup_steps=CFG['warmup_steps'],
    weight_decay=0.01,
    fp16=True,
    predict_with_generate=True,
    generation_max_length=CFG['max_target_length'],  # ← fixes max_length=20 bug
    generation_num_beams=CFG['num_beams'],
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='rougeL_f1',
    greater_is_better=True,
    logging_steps=100,
    seed=CFG['seed'],
    report_to='none',
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    data_collator=collator,
    compute_metrics=compute_metrics,
)

trainer.train()
trainer.save_model(CFG['output_dir'])
tokenizer.save_pretrained(CFG['output_dir'])
print('Training complete. Model saved.')

## Step 8 — Validation ROUGE by Subset

In [ ]:
from tqdm import tqdm
from rouge_score import rouge_scorer as rs

model.eval()
model.to(device)

def generate_batch(prompts, batch_size=8):
    preds = []
    for i in tqdm(range(0, len(prompts), batch_size)):
        batch = prompts[i:i+batch_size]
        inputs = tokenizer(
            batch, return_tensors='pt',
            max_length=CFG['max_input_length'],
            truncation=True, padding=True
        ).to(device)
        with torch.no_grad():
            out = model.generate(
                **inputs,
                max_new_tokens=CFG['max_target_length'],  # ← key fix
                num_beams=CFG['num_beams'],
                no_repeat_ngram_size=3,
                early_stopping=True,
            )
        preds.extend(tokenizer.batch_decode(out, skip_special_tokens=True))
    return preds

val_preds = generate_batch(val_df['prompt'].tolist())

# overall ROUGE
scorer = rs.RougeScorer(['rouge1', 'rougeL'], use_stemmer=False)
r1, rl = [], []
for p, ref in zip(val_preds, val_df['output'].tolist()):
    s = scorer.score(ref, p)
    r1.append(s['rouge1'].fmeasure)
    rl.append(s['rougeL'].fmeasure)
print(f'Val ROUGE-1: {sum(r1)/len(r1):.4f} | ROUGE-L: {sum(rl)/len(rl):.4f}')

# per subset
from collections import defaultdict
sub_scores = defaultdict(lambda: {'r1':[], 'rl':[]})
for p, ref, sub in zip(val_preds, val_df['output'].tolist(), val_df['subset'].tolist()):
    s = scorer.score(ref, p)
    sub_scores[sub]['r1'].append(s['rouge1'].fmeasure)
    sub_scores[sub]['rl'].append(s['rougeL'].fmeasure)

rows = []
for sub, v in sorted(sub_scores.items()):
    rows.append({'subset': sub,
                 'ROUGE-1': round(sum(v['r1'])/len(v['r1']),4),
                 'ROUGE-L': round(sum(v['rl'])/len(v['rl']),4)})
print(pd.DataFrame(rows).to_string(index=False))

## Step 9 — Generate Test Predictions and Save Submission

In [ ]:
test_preds = generate_batch(test_df['prompt'].tolist())

# sanity check predictions
print('Sample predictions:')
for i in range(3):
    print(f'  [{test_df["subset"].iloc[i]}] Q: {test_df["input"].iloc[i][:80]}')
    print(f'  A: {test_preds[i][:150]}')
    print()

submission = pd.DataFrame({
    'ID'         : test_df['ID'],
    'TargetRLF1' : test_preds,
    'TargetR1F1' : test_preds,
    'TargetLLM'  : test_preds,
})
submission.to_csv(CFG['submission_path'], index=False)
print(f'Saved {len(submission)} rows to {CFG["submission_path"]}')
submission.head(3)

## Step 10 — Download Submission

In [ ]:
from google.colab import files
files.download(CFG['submission_path'])